# ERA5-Land: weather windows for fire-prevention operations

**Business question.** When should field patrols, water-management teams, and public-alert campaigns be scaled up? This notebook uses only ERA5-Land weather data (rainfall, air temperature, and soil moisture)—not NASA FIRMS or population data.

**Decision use.** Identify recurring dry windows, quantify their intensity, and translate them into staffing and prevention-readiness triggers.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'data').exists())
files = sorted((ROOT / 'data/interim/era5_daily_parts').glob('era5_daily_*.parquet'))
assert files, 'ERA5-Land daily extracts were not found.'

daily_parts, grid_month_parts = [], []
for path in files:
    frame = pd.read_parquet(path, columns=['date', 'grid_id', 'rainfall_mm', 'temperature_mean_c', 'soil_moisture'])
    daily_parts.append(frame.groupby('date', as_index=False).mean(numeric_only=True))
    frame['month_start'] = pd.to_datetime(frame['date']).dt.to_period('M').dt.to_timestamp()
    grid_month_parts.append(frame.groupby(['grid_id', 'month_start'], as_index=False).agg(
        rainfall_mm=('rainfall_mm', 'sum'), soil_moisture=('soil_moisture', 'mean')
    ))

weather = pd.concat(daily_parts, ignore_index=True).drop_duplicates('date').sort_values('date')
weather['date'] = pd.to_datetime(weather['date'])
weather['month'] = weather['date'].dt.month
weather['year'] = weather['date'].dt.year
grid_month = pd.concat(grid_month_parts, ignore_index=True)
weather.head()

In [ ]:
monthly_series = weather.set_index('date').resample('MS').agg(
    rainfall_mm=('rainfall_mm', 'sum'), temperature_c=('temperature_mean_c', 'mean'), soil_moisture=('soil_moisture', 'mean')
)
climate = monthly_series.groupby(monthly_series.index.month).mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']
climate['dryness_score'] = climate['rainfall_mm'].rank(pct=True) + climate['soil_moisture'].rank(pct=True)
watch_months = climate.nsmallest(4, 'dryness_score').index.tolist()

# Compare each grid with its own history: this avoids calling naturally drier places 'dry' merely because of geography.
grid_month['rain_rank'] = grid_month.groupby('grid_id')['rainfall_mm'].rank(pct=True)
grid_month['soil_rank'] = grid_month.groupby('grid_id')['soil_moisture'].rank(pct=True)
grid_month['compound_dry'] = (grid_month['rain_rank'] <= .25) & (grid_month['soil_rank'] <= .25)
coverage = grid_month.groupby('month_start')['compound_dry'].mean().mul(100)
heatmap = coverage.rename_axis('month_start').reset_index(name='share').assign(
    year=lambda x: x.month_start.dt.year, month=lambda x: x.month_start.dt.month
).pivot(index='year', columns='month', values='share').reindex(columns=range(1, 13))

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
axes[0].bar(climate.index, climate['rainfall_mm'], color='#2878b5')
for month in watch_months: axes[0].axvspan(month - .45, month + .45, color='#c94c4c', alpha=.12)
axes[0].set(title='Klimatologi hujan dan jendela siaga', xlabel='Month', ylabel='Average monthly rainfall (mm)', xticks=range(1, 13), xticklabels=month_names)
image = axes[1].imshow(heatmap, cmap='YlOrRd', aspect='auto', vmin=0)
axes[1].set(title='Cakupan kekeringan majemuk', xlabel='Month', ylabel='Year', xticks=range(12), xticklabels=month_names, yticks=range(len(heatmap.index)), yticklabels=heatmap.index)
fig.colorbar(image, ax=axes[1], label='% grid at own lower-quartile rain and soil moisture')
plt.show()

widest = coverage.idxmax()
print(f'Jendela pre-positioning: {[month_names[m-1] for m in watch_months]}.')
print(f'Kekeringan majemuk paling luas terjadi pada {widest:%b %Y}: {coverage.loc[widest]:.1f}% grid berada pada kuartil hujan dan kelembapan tanah terendahnya.')
print('Implikasi karhutla: saat cakupan kondisi kering meluas, audit muka air gambut, kesiapan pompa, larangan pembakaran, dan koordinasi lintas wilayah perlu dinaikkan sebelum laporan api membesar. Ini adalah indikator kesiapsiagaan, bukan bukti kebakaran atau prediksi kejadian.')

In [ ]:
# EDA per provinsi: kalender kesiapsiagaan tidak dipukul rata untuk seluruh pulau.
grid_province = pd.read_parquet(ROOT / 'data/processed/dim_grid.parquet', columns=['grid_id', 'province']).drop_duplicates('grid_id')
province_month = grid_month.merge(grid_province, on='grid_id', how='inner', validate='many_to_one')
assert province_month['province'].nunique() == 5
province_climate = province_month.assign(month=province_month['month_start'].dt.month).groupby(['province', 'month'], as_index=False).agg(
    rainfall_mm=('rainfall_mm', 'mean'), soil_moisture=('soil_moisture', 'mean')
)
province_climate['rain_rank'] = province_climate.groupby('province')['rainfall_mm'].rank(pct=True)
province_climate['soil_rank'] = province_climate.groupby('province')['soil_moisture'].rank(pct=True)
province_climate['dryness_score'] = province_climate['rain_rank'] + province_climate['soil_rank']
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
for province, part in province_climate.groupby('province'):
    axes[0].plot(part['month'], part['rainfall_mm'], marker='o', label=province)
    axes[1].plot(part['month'], part['soil_moisture'], marker='o', label=province)
for ax, title, ylabel in zip(axes, ['Musim hujan per provinsi', 'Kelembapan tanah per provinsi'], ['Hujan rata-rata bulanan (mm)', 'Kelembapan tanah (m³/m³)']):
    ax.set(title=title, xlabel='Bulan', ylabel=ylabel, xticks=range(1, 13), xticklabels=month_names); ax.grid(alpha=.25)
axes[1].legend(fontsize=8); plt.show()
windows = province_climate.sort_values('dryness_score').groupby('province').head(3).groupby('province')['month'].apply(lambda months: ', '.join(month_names[m - 1] for m in sorted(months)))
print('Tiga bulan siaga relatif per provinsi:')
print(windows.to_string())
print('Implikasi bisnis: kalender regional dipakai untuk koordinasi, tetapi penempatan pompa, patroli, dan audit kanal mengikuti jendela lokal per provinsi.')

## Decision notes

- Hujan rendah saja belum cukup menjelaskan kesiapan karhutla: kelembapan tanah adalah indikator tambahan untuk melihat pengeringan yang tersimpan.
- Bulan siaga dipakai untuk *pre-positioning*, bukan untuk menyimpulkan kebakaran pasti terjadi. Trigger harian/mingguan tetap harus menggunakan pembaruan ERA5 dan observasi lokal.
- Ketika cakupan kekeringan majemuk lintas grid meningkat, gunakan sebagai alasan memperkuat koordinasi lintas provinsi, audit muka air gambut, kesiapan pompa, dan pengawasan pembakaran lahan.
- Gunakan tiga bulan siaga relatif per provinsi sebagai kalender *pre-positioning* lokal; jangan memaksakan satu ambang tanggal Kalimantan untuk seluruh provinsi.